Ingtestion

In [1]:
from langchain_core.documents import Document

doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Ritesh Patel",
        "date_created":"2025-01-01"
    }
)
doc

#Importance of meta data: to apply filters

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Ritesh Patel', 'date_created': '2025-01-01'}, page_content='this is the main text content I am using to create RAG')

In [2]:
# creating text files or you can also create manuall. 
import os
os.makedirs("data/text_files", exist_ok=True)

sample_texts = {
    "data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
"""
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [3]:
# reading it using text loader.

from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "data/text_files/python_intro.txt",
    encoding="utf-8"
)

document = loader.load()

print(document)

C:\Users\rites\AppData\Local\Temp\ipykernel_3920\745951883.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [4]:
# or # reading it using directory loader.

from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",  # Pattern to match files
    loader_cls=TextLoader,  # Loader class to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

docu = dir_loader.load()
docu

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n'),
 Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability. Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming lan

In [5]:
### Load all the PDF files from the directory

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [6]:
### Read all the PDFs inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information from myself to metadata for better chunking
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents) # storing it inside this variable.

            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

# Process all PDFs in the data/pdf directory
all_pdf_documents = process_all_pdfs("data/pdf")
all_pdf_documents

Found 1 PDF files to process

Processing: ritesh_AI.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-23T17:23:18+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-23T17:23:18+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\ritesh_AI.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ritesh_AI.pdf', 'file_type': 'pdf'}, page_content='Ritesh Patel\n+91 8858295418|riteshpatel1884@gmail.com |linkedin/riteshpatel1884 |github/riteshpatel1884\nEducation\nKIET Deemed to be UniversityGhaziabad, India\nB.Tech in Computer Science — 7.48 GPA Oct 2023 - June 2027\nRani Laxmi Bai Memorial SchoolLucknow, India\nClass XII (ISC) — 84% 2021-2022\nRani Laxmi Bai Memorial SchoolLucknow, India\nClass X (ICSE) — 86% 2019-2020\nExperience\nAirkrit India|Internship|Data AnalystMay - Aug 2026\n• Built data preprocessing pipelines (Python,

Chunking

In [7]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...") #  first 200 characters will be shown
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [8]:
chunks = split_documents(all_pdf_documents)

Split 1 documents into 4 chunks

Example chunk:
Content: Ritesh Patel
+91 8858295418|riteshpatel1884@gmail.com |linkedin/riteshpatel1884 |github/riteshpatel1884
Education
KIET Deemed to be UniversityGhaziabad, India
B.Tech in Computer Science — 7.48 GPA Oct...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-23T17:23:18+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-23T17:23:18+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\ritesh_AI.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ritesh_AI.pdf', 'file_type': 'pdf'}


Embeddings

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid  # so that every record we are instering in the vector database has a unique id.
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()  # it is the protected method which should be called only inside this class.

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
    

# Initializing the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3758.08it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\rites\AppData\Local\Temp\ipykernel_3920\2204422507.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


Vector Store

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./data/vector_store"  # means whatever vector store we are creating, it will be stored in this directory.
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                    }
                )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add documents to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise
vectorStore = VectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


Existing documents in collection: 0 becoz we have not added any documents yet. We will add the documents after generating embeddings for the chunks we created from the PDFs.

In [12]:
# convert the text(chunks) in embeddings and store it in the vector store.

texts = [doc.page_content for doc in chunks]

### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the vector database
vectorStore.add_documents(chunks, embeddings)

Generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to vector store...
Successfully added 4 documents to vector store
Total documents in collection: 4


Retriever pipeline from vector store

In [13]:
# This retriever is based on the top of vector store which absed on every query we will get it will give the response.
# First query will be converted into embedding and then it will be compared with the embeddings stored in the vector store and based on the similarity score it will return the top n documents.

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} documents "
                    f"(after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

retriever = RAGRetriever(vectorStore, embedding_manager)
retriever

In [16]:
retriever.retrieve("What role i am targetting about?")

Retrieving documents for query: 'What role i am targetting about?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_78d3b779_0',
  'content': 'Ritesh Patel\n+91 8858295418|riteshpatel1884@gmail.com |linkedin/riteshpatel1884 |github/riteshpatel1884\nEducation\nKIET Deemed to be UniversityGhaziabad, India\nB.Tech in Computer Science — 7.48 GPA Oct 2023 - June 2027\nRani Laxmi Bai Memorial SchoolLucknow, India\nClass XII (ISC) — 84% 2021-2022\nRani Laxmi Bai Memorial SchoolLucknow, India\nClass X (ICSE) — 86% 2019-2020\nExperience\nAirkrit India|Internship|Data AnalystMay - Aug 2026\n• Built data preprocessing pipelines (Python, Pandas, NumPy) on 10K+ records, applying cleaning, feature\nengineering, and anomaly detection techniques\n• Applied statistical analysis and pattern detection to identify sales and customer behavior trends, feeding into 15+\nvisualizations for stakeholder decision-making\n• Automated report generation using LLM-based summarization, reducing manual reporting time by 30%\nProjects\nLLM Evaluation Platform|FastAPI, Next.js, LLMs, PostgreSQL, Groq APIAug 2026 - Presen

LLM Integration

In [24]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [25]:
def rag_simple(query, retriever, llm, top_k=3):

    # 1. Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    # 2. Build context
    context = "\n\n".join(
        doc["content"]
        for doc in results
    ) if results else ""

    # 3. Stop if nothing relevant was retrieved
    if not context:
        return "No relevant context found to answer the question."

    # 4. Create prompt
    prompt = f"""
Use the following context to answer the question.
Answer only from the provided context.
If the answer is not present in the context, say:
"I don't have enough information in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # 5. Generate answer
    response = llm.invoke(prompt.format(query=query, context=context))

    # 6. Return answer text
    return response.content

In [28]:
answer = rag_simple(
    "What projects has Ritesh built?",
    retriever,
    llm,
    top_k=3
)

print(answer)

Retrieving documents for query: 'What projects has Ritesh built?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.05it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Ritesh has built the following projects:

1. **LLM Evaluation Platform** – Developed with FastAPI, Next.js, LLMs, PostgreSQL, and the Groq API (started Aug 2026 – present).  

2. **Job Application Tracker & Rejection Analytics** – Built using Next.js, JavaScript, and PostgreSQL (Mar 2026 – May 2026).


In [30]:
answer = rag_simple(
    "What are his skills",
    retriever,
    llm,
    top_k=3
)

print(answer)

Retrieving documents for query: 'What are his skills'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Technical Skills**

- **Languages:** Python, Java, JavaScript, SQL  
- **AI/ML:** Large Language Models (LLMs), Retrieval‑Augmented Generation (RAG), Prompt Engineering, Vector Search, Model Evaluation, PyTorch, Hugging Face  
- **Frameworks:** LangChain, LangGraph, FastAPI, Next.js  
- **Databases & Tools:** PostgreSQL, MongoDB, Docker, Git, GitHub, OpenAI, Gemini


Enhanced RAG pipeline

In [32]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(
    query,
    retriever,
    llm,
    top_k=5,
    min_score=0.2,
    return_context=False
):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join(
        [doc['content'] for doc in results]
    )

    sources = [
        {
            'source': doc['metadata'].get(
                'source_file',
                doc['metadata'].get('source', 'unknown')
            ),
            'page': doc['metadata'].get('page', 'unknown'),
            'score': doc['similarity_score'],
            'preview': doc['content'][:120] + '...'
        }
        for doc in results
    ]

    confidence = max(
        [doc['similarity_score'] for doc in results]
    )

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(
        [prompt.format(context=context, query=query)]
    )

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context

    return output


# Example usage:
result = rag_advanced(
    "What are my skills?",
    retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What are my skills?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Technical Skills**

- **Programming Languages:** Python, Java, JavaScript, SQL  
- **AI/ML:** Large Language Models (LLMs), Retrieval‑Augmented Generation (RAG), Prompt Engineering, Vector Search, Model Evaluation, PyTorch, Hugging Face  
- **Frameworks & Libraries:** LangChain, LangGraph, FastAPI, Next.js  
- **Data & DevOps Tools:** PostgreSQL, MongoDB, Docker, Git/GitHub, Pandas, NumPy  
- **Cloud & Services:** Oracle Cloud Infrastructure, OpenAI API, Gemini, Groq API  

**Additional Competencies**

- Data preprocessing, feature engineering, statistical analysis, anomaly detection  
- Building dashboards and visual analytics (charts/graphs)  
- Designing RAG‑based Text‑to‑SQL systems, ambiguity detection, multi‑turn clarification workflows  
- Automating report generation with LLM‑based summarization  

**Certifications**

- Oracle Cloud Infrastructure 2025 Certified  
- Generative AI Professional (certified Sep 2025)
Sources: [{'source': 'ritesh_AI.pdf', 'page': 0, 'score

Refer src folder for making these codes into modular 

D:\GenAI\07_RAG\04_rag_revision\src